# NB08bis — Abundance calibration and spatial/temporal context
Exploratory sensitivity experiments only, to inform final NB08. CPU. Uses completed NB08 V5 data and exact outer folds; no LOFO, no legacy data, no production refit. All new outputs in 08bis_abundance_exploration_v5. Run sequentially; per-arm/fold and extraction caches support restart. Fixed sensitivity recipes, not nested hyperparameter selection; outer model comparisons exploratory. Optional failures logged rather than treated as negative findings.

**Historical recovery extension:** Sections 08–09 added after the original exploration. Existing cells 00–07 remain intact. On a fresh session run 00–01, then 03 with RUN_SCALE=False, then 08–09. On a live session with those definitions loaded, append/run only the new cells. No need to repeat HLS, NICFI or context extraction.

**Final calibration extension:** Section 10 is independent of 02–09 once their caches exist. Fresh session: execute 00 → 01 → 10 only. No new downloads. Do not rerun the historical model grid.


In [ ]:
%pip -q install earthengine-api rasterio geopandas pyarrow scikit-learn scipy pandas matplotlib requests joblib
from pathlib import Path
from contextlib import ExitStack
from datetime import datetime,timezone
from collections import defaultdict
import os,json,hashlib,time,math,requests
import numpy as np,pandas as pd,geopandas as gpd,matplotlib.pyplot as plt
import rasterio
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from rasterio.transform import from_origin,Affine
from rasterio.io import MemoryFile
from rasterio.warp import transform_bounds
from shapely.geometry import box
from shapely.ops import unary_union
from shapely.prepared import prep
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import roc_auc_score,average_precision_score
from scipy.stats import spearmanr
from threadpoolctl import threadpool_limits
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():drive.mount('/content/drive')
threadpool_limits(limits=1)
V5_ROOT=next((p for p in (Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final/V5'),Path('/content/drive/MyDrive/Musanga/Final/V5')) if (p.parent/'Data').is_dir() and p.is_dir()),None)
assert V5_ROOT is not None
ROOT=V5_ROOT/'Outputs/08_satellite_reference';ROOT.mkdir(parents=True,exist_ok=True)
OUT=V5_ROOT/'Outputs/08_satellite_science_v5';OUT.mkdir(parents=True,exist_ok=True)
V4_OUT=OUT # compatibility variable for retained analysis functions; never a V4 path
NB07=V5_ROOT/'Outputs/07_RGB_unified_production_v5'
MANIFEST=NB07/'07_production_mosaic_manifest.json'
assert MANIFEST.is_file(),'NB07 must finish all six mosaics before NB08 execution.'
M=json.loads(MANIFEST.read_text())
assert M['complete'] and M['schema']=='musanga_v5_nb07_rgb_mosaics' and len(M['mosaics'])==6
SITES=('Baego','MOS','Yangambi','Yoko')
YEARS={'Baego':2023,'MOS':2024,'Yangambi':2024,'Yoko':2020}
KEYS=['site','mosaic','row','col']
CRS='EPSG:32635';TILE_M=1500;LATTICE_M=1;MIN_SUPPORT=.90
SCALES=[10,25,30,50,75,100]
EE_PROJECT='musanga-511110' # change to an authorized project when transferring
S2_SOURCE='COPERNICUS/S2_SR_HARMONIZED'
EMB_SOURCE='GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL'
S2_BANDS=['B2','B3','B4','B8','B5','B6','B7','B8A','B11','B12']
SCL_REJECT=[0,1,3,7,8,9,10,11]
INDEX_NAMES=['NDVI','NDMI','NBR','NDRE']
EMB_BANDS=[f'A{i:02d}' for i in range(64)]
ANNUAL=[f'RAW_ANNUAL__{i:02d}' for i in range(1,16)]
QUARTER=[f'S2_QUARTERLY_3YR__{i:02d}' for i in range(1,45)]
EMB=[f'EMB_ANNUAL__{i:02d}' for i in range(1,65)]
Q_COUNTS=[QUARTER[i-1] for i in (11,22,33,44)]
S2=ANNUAL+[c for c in QUARTER if c not in Q_COUNTS]
FEATURE_SETS={'M1_S2':S2,'M2_EMB':EMB,'M3_S2_EMB':S2+EMB}
ALL_FEATURES=sorted(set(sum(FEATURE_SETS.values(),[])))
SEED=20261120;SEED17=SEED;CAP_PER_TRAIN_FOREST=1500
MODEL=dict(learning_rate=.06,max_iter=180,max_leaf_nodes=20,min_samples_leaf=35,l2_regularization=2,early_stopping=False)
THRESHOLDS=(.05,.20,.40)
RUN_STRUCTURE=True  # supplementary; independent GEE assets may require access
RUN_ABUNDANCE_SENSITIVITY=True
# Normal restart: reuse completed data and analyses. True rebuilds derived aggregates/results;
# validated per-tile imagery and UAV tiles remain reusable.
FORCE_REBUILD=False
def digest(obj):return hashlib.sha256(json.dumps(obj,sort_keys=True,default=str).encode()).hexdigest()
def atomic_json(p,obj):
 p=Path(p);tmp=p.with_suffix(p.suffix+'.partial');tmp.write_text(json.dumps(obj,indent=2,default=str));os.replace(tmp,p)
def atomic_parquet(d,p):
 p=Path(p);tmp=p.with_suffix('.partial');d.to_parquet(tmp,index=False);os.replace(tmp,p)
def dataframe_sha(d):return hashlib.sha256(pd.util.hash_pandas_object(d,index=False).values.tobytes()).hexdigest()

from scipy.spatial import cKDTree
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn import __version__ as SK_VERSION
from scipy.ndimage import uniform_filter
NB08_OUT=OUT
OUT=V5_ROOT/'Outputs/08bis_abundance_exploration_v5';OUT.mkdir(exist_ok=True)
RUN_SCALE=True;RUN_CONTEXT=True;RUN_NICFI=True;RUN_HLS=True
BASE=S2+EMB;BASE_NAME='M3_S2_EMB__HGB__forest_equal'
candidates=[]
for p in sorted(ROOT.glob('native_50m_reference_*.parquet')):
 sig=p.stem.removeprefix('native_50m_reference_');cf=NB08_OUT/sig/'config.json'
 if not cf.is_file():continue
 c=json.loads(cf.read_text());tc=ROOT/'uav_targets'/c['target_id']/'config.json'
 if not tc.is_file() or digest(c)[:20]!=sig or c['features']!=FEATURE_SETS:continue
 t=json.loads(tc.read_text())
 if t['model']!=M['model_sha'] or t['policy']!=M['policy_sha'] or t['support']!=MIN_SUPPORT:continue
 old={a['name']:a for a in t['mosaics']}
 if set(old)!={a['name'] for a in M['mosaics']}:continue
 valid=True
 for a in M['mosaics']:
  f=NB07/'reconstructed_mosaics'/Path(a['output']).name
  if not f.is_file() or old[a['name']]['digest']!=a['digest'] or old[a['name']]['size']!=f.stat().st_size:valid=False;break
  with rasterio.open(f) as ds:
   if ds.tags().get('signature')!=a['digest'] or list(ds.bounds)!=old[a['name']]['bounds']:valid=False;break
 if not valid:continue
 d=pd.read_parquet(p)
 if dataframe_sha(d[KEYS+['x','y','target_fraction']+ALL_FEATURES])==c['source_sha256']:candidates.append((p,c,d))
assert candidates,'Complete NB08 reference first'
assert len({c['source_sha256'] for _,c,_ in candidates})==1,'Multiple distinct references; select explicitly'
REF,CONFIG,D=candidates[0];SOURCE_SHA=CONFIG['source_sha256']
SAT_ID=CONFIG['satellite_id'];SAT_DIR=ROOT/'satellite_10m'/SAT_ID
TARGET_DIR=ROOT/'uav_targets'/CONFIG['target_id']
assert len(D)==CONFIG['n'] and set(D.site)==set(SITES) and not D.duplicated(['site','row','col']).any()
assert D.target_fraction.between(0,1).all() and np.isfinite(D[BASE].to_numpy(float)).all()
found=[]
for p in sorted((NB08_OUT/'within_landscape_50m').glob('*/heldout_predictions.parquet')):
 c=json.loads((p.parent/'config.json').read_text());z=pd.read_parquet(p)
 if c.get('version')!='within_landscape_v2' or c.get('buffer')!=100 or c.get('block')!=500 or BASE_NAME not in z:continue
 cols=['site','row','col','x','y','target_fraction']
 if not z[cols].equals(D[cols]):continue
 used=sorted(set(sum(c['features'].values(),[])))
 if dataframe_sha(D.assign(fold21=z.fold21.to_numpy())[['site','x','y','target_fraction','fold21']+used])==c['source']:found.append((p,z))
assert found,'Complete NB08 spatial holdout first'
BASE_P,OLD=found[0];M21=D.assign(block21=OLD.block21.to_numpy(),fold21=OLD.fold21.to_numpy())
mapping=M21.groupby('block21').fold21.first().to_dict()
assert M21.groupby('block21').fold21.nunique().eq(1).all() and M21.groupby('site').fold21.nunique().eq(5).all()
RUN=OUT/digest(dict(version='08bis_v1',source=SOURCE_SHA,fold=dataframe_sha(M21[['site','row','col','fold21']]),sklearn=SK_VERSION))[:20];RUN.mkdir(exist_ok=True)
atomic_json(RUN/'input_contract.json',dict(reference=str(REF),source=SOURCE_SHA,nb08_predictions=str(BASE_P),satellite=SAT_ID,sklearn=SK_VERSION))
REPORTS=[];SEED21=20261123
print('Reference:',REF,'\nPredictions:',BASE_P,'\nOutput:',RUN)
display(M21.groupby(['site','fold21']).size().unstack())


## 01 — Shared evaluator
Natural validation distribution. Training-only 10 pp abundance bins, capped inverse-frequency or inverse-square-root weights, optionally equal total forest weights. Fixed baseline and flexible HGB settings. Ordinal model integrates ten exceedance probabilities. 100 m centre exclusion baseline, stricter buffer for context and larger cells. Report pooled R², mean site R², within-forest R² (no between-forest variance), calibration slope, high (≥60%) and low (≤10%) bias, bin errors and paired block bootstrap. No automated outer-fold tuning.


In [ ]:
FLEX=dict(learning_rate=.06,max_iter=300,max_leaf_nodes=63,min_samples_leaf=15,l2_regularization=.5,early_stopping=False)
def bins(y):return np.clip(np.floor(np.asarray(y)*10).astype(int),0,9)
def weights(g,mode):
 b=bins(g.target_fraction);w=np.ones(len(g))
 if mode=='natural':return w
 if mode.startswith('forest'):
  for s in g.site.unique():
   m=g.site.eq(s).to_numpy();a=np.ones(m.sum())
   if 'bins' in mode:
    a=np.bincount(b[m],minlength=10)[b[m]].astype(float)**(-(.5 if 'soft' in mode else 1));a/=a.mean();a=np.clip(a,.2,5)
   w[m]=a/a.sum()
 elif mode.startswith('global'):
  w=np.bincount(b,minlength=10)[b].astype(float)**(-(.5 if 'soft' in mode else 1));w/=w.mean();w=np.clip(w,.2,5)
 else:raise ValueError(mode)
 return w/w.mean()
def splits(d,buffer):
 xy=d[['x','y']].to_numpy();site=d.site.to_numpy();fold=d.fold21.to_numpy()
 for k in range(5):
  te=np.flatnonzero(fold==k);tr=np.flatnonzero(fold!=k);keep=np.ones(len(tr),bool)
  for s in SITES:
   a=np.flatnonzero(site[tr]==s);b=te[site[te]==s]
   if len(a) and len(b):keep[a]=cKDTree(xy[b]).query(xy[tr[a]])[0]>=buffer
  assert len(te) and keep.any()
  yield k,tr[keep],te
def fit(d,tr,te,f,kind,w,p):
 g=d.iloc[tr];X=g[f].to_numpy(np.float32);Xt=d.iloc[te][f].to_numpy(np.float32);y=g.target_fraction.to_numpy();sw=weights(g,w)
 if kind=='ordinal':
  pp=[]
  for t in np.arange(.05,1,.1):
   z=(y>t).astype(int)
   if len(np.unique(z))==1:pp.append(np.full(len(te),z[0]));continue
   m=HistGradientBoostingClassifier(random_state=SEED21,**p);m.fit(X,z,sample_weight=sw);pp.append(m.predict_proba(Xt)[:,1])
  return np.sort(np.stack(pp,axis=1),axis=1)[:,::-1].mean(axis=1)
 m=HistGradientBoostingRegressor(random_state=SEED21,**p);m.fit(X,y,sample_weight=sw);return np.clip(m.predict(Xt),0,1)
def stats(y,p):
 y=np.asarray(y,float);p=np.asarray(p,float);e=p-y;h=y>=.6;l=y<=.1
 return dict(n=len(y),R2=1-np.mean(e**2)/np.var(y) if np.var(y)>0 else np.nan,MAE_pp=100*np.mean(abs(e)),RMSE_pp=100*np.sqrt(np.mean(e**2)),bias_pp=100*e.mean(),slope=np.mean((y-y.mean())*(p-p.mean()))/np.var(y) if np.var(y)>0 else np.nan,high_n=int(h.sum()),high_bias_pp=100*e[h].mean() if h.any() else np.nan,high_MAE_pp=100*abs(e[h]).mean() if h.any() else np.nan,low_n=int(l.sum()),low_bias_pp=100*e[l].mean() if l.any() else np.nan)
def arm(name,f=BASE,kind='reg',w='forest',p=MODEL):return (name,list(f),kind,w,dict(p))
def evaluate(d,arms,label,buffer=100):
 d=d.reset_index(drop=True);assert set(d.site)==set(SITES) and d.groupby('site').fold21.nunique().eq(5).all()
 fields=sorted({c for _,f,_,_,_ in arms for c in f});assert np.isfinite(d[fields].to_numpy(float)).all()
 cols=['site','row','col','x','y','target_fraction','block21','fold21']
 cfg=dict(version='08bis_cv_v1',source=dataframe_sha(d[cols+fields]),arms=arms,buffer=buffer,seed=SEED21,sklearn=SK_VERSION)
 folder=RUN/label/digest(cfg)[:20];folder.mkdir(parents=True,exist_ok=True);atomic_json(folder/'config.json',cfg)
 names=[a[0] for a in arms];pred=d[cols].copy();final=folder/'predictions.parquet'
 if final.is_file() and not FORCE_REBUILD:
  q=pd.read_parquet(final);assert q[cols].equals(pred) and np.isfinite(q[names].to_numpy(float)).all();pred=q;print('MODEL CACHE',label)
 else:
  for k,tr,te in splits(d,buffer):
   for name,f,kind,w,p in arms:
    cache=folder/f'{name}_fold{k}.npz'
    if cache.is_file() and not FORCE_REBUILD:
     with np.load(cache,allow_pickle=False) as z:assert np.array_equal(z['test'],te);v=z['pred']
    elif w=='local':
     v=np.full(len(te),np.nan)
     for s in SITES:
      a=tr[d.site.iloc[tr].eq(s).to_numpy()];b=np.flatnonzero(d.site.iloc[te].eq(s).to_numpy())
      if len(b):assert len(a);v[b]=fit(d,a,te[b],f,kind,'natural',p)
    else:v=fit(d,tr,te,f,kind,w,p)
    assert np.isfinite(v).all() and len(v)==len(te)
    if not cache.is_file() or FORCE_REBUILD:
     tmp=cache.with_suffix('.partial')
     with open(tmp,'wb') as out:np.savez_compressed(out,test=te,pred=v)
     os.replace(tmp,cache)
    pred.loc[te,name]=v;print(label,k,name,flush=True)
  assert pred[names].notna().all().all();atomic_parquet(pred,final)
 rows=[];summ=[];cal=[];ci=[]
 for name in names:
  for s,g in pred.groupby('site'):
   rows.append(dict(model=name,site=s,**stats(g.target_fraction,g[name])))
   for b,q in g.assign(bin=bins(g.target_fraction)).groupby('bin'):
    cal.append(dict(model=name,site=s,bin=b,observed_pp=100*q.target_fraction.mean(),predicted_pp=100*q[name].mean(),**stats(q.target_fraction,q[name])))
  a=pd.DataFrame(rows);a=a[a.model.eq(name)];mos=a[a.site.eq('MOS')].iloc[0];u=stats(pred.target_fraction,pred[name])
  den=np.mean([g.target_fraction.var(ddof=0) for _,g in pred.groupby('site')])
  summ.append(dict(model=name,n=len(pred),pooled_R2=u['R2'],pooled_MAE_pp=u['MAE_pp'],forest_MAE_pp=a.MAE_pp.mean(),mean_site_R2=a.R2.mean(),within_forest_R2=1-np.mean((a.RMSE_pp/100)**2)/den,mean_site_slope=a.slope.mean(),MOS_R2=mos.R2,MOS_MAE_pp=mos.MAE_pp,MOS_slope=mos.slope,MOS_high_bias_pp=mos.high_bias_pp,MOS_low_bias_pp=mos.low_bias_pp))
  rng=np.random.default_rng(SEED21);dist=[]
  for _,g in pred.groupby('site'):
   q=g.assign(delta=100*(abs(g[name]-g.target_fraction)-abs(g[names[0]]-g.target_fraction))).groupby('block21').delta.agg(['sum','count']).to_numpy()
   draws=q[rng.integers(0,len(q),(500,len(q)))].sum(axis=1);dist.append(draws[:,0]/draws[:,1])
  z=np.mean(dist,axis=0);ci.append(dict(model=name,baseline=names[0],ci_low=np.quantile(z,.025),ci_high=np.quantile(z,.975)))
 sm=pd.DataFrame(summ).sort_values('forest_MAE_pp');per=pd.DataFrame(rows);cb=pd.DataFrame(cal)
 sm.to_csv(folder/'summary.csv',index=False);per.to_csv(folder/'by_forest.csv',index=False);cb.to_csv(folder/'abundance_bins.csv',index=False);pd.DataFrame(ci).to_csv(folder/'paired_bootstrap.csv',index=False)
 display(sm.round(3));display(per[per.site.eq('MOS')].round(3));REPORTS.append(dict(stage=label,path=str(folder),n=len(pred),status='complete'))
 # Figures are derived from cached predictions and saved separately.
 chosen=list(dict.fromkeys([names[0],*sm.model.iloc[:3]]))
 fig,axs=plt.subplots(2,len(chosen),figsize=(4*len(chosen),8),squeeze=False,layout='constrained')
 for j,name in enumerate(chosen):
  for i,g in enumerate([pred,pred[pred.site.eq('MOS')]]):
   ax=axs[i,j];ax.hexbin(100*g.target_fraction,100*g[name],gridsize=45,bins='log',mincnt=1,extent=(0,100,0,100));ax.plot([0,100],[0,100],'k--');ax.set(xlim=(0,100),ylim=(0,100),xlabel='UAV cover (%)',ylabel='Prediction (%)',title=('Pooled' if i==0 else 'MOS')+' | '+name,aspect='equal')
 fig.savefig(folder/'pooled_MOS.png',dpi=150);plt.show();plt.close(fig)
 fig,axs=plt.subplots(2,2,figsize=(10,8),layout='constrained')
 for s,ax in zip(SITES,axs.ravel()):
  for name in chosen:
   q=cb[(cb.site==s)&(cb.model==name)];ax.plot(q.observed_pp,q.predicted_pp,'o-',label=name,ms=3)
  ax.plot([0,100],[0,100],'k--');ax.set(xlim=(0,100),ylim=(0,100),title=s,xlabel='Observed bin mean (%)',ylabel='Predicted bin mean (%)')
 axs[0,0].legend(fontsize=7);fig.savefig(folder/'calibration.png',dpi=150);plt.show();plt.close(fig)
 return pred,sm
def optional(label,fn):
 try:return fn()
 except Exception as e:
  REPORTS.append(dict(stage=label,status='failed',error=repr(e)));print(label,'FAILED:',repr(e),flush=True);return None


## 02 — Abundance weighting and flexibility at 50 m
Fixed sensitivity arms, including two ordinal variants. Baseline verification against NB08.

In [ ]:
a=[arm('baseline'),arm('natural',w='natural'),arm('local',w='local'),arm('global_bins',w='global_bins'),arm('global_bins_soft',w='global_bins_soft'),arm('forest_bins',w='forest_bins'),arm('forest_bins_soft',w='forest_bins_soft'),arm('flex_forest',p=FLEX),arm('flex_natural',w='natural',p=FLEX),arm('flex_bins_soft',w='forest_bins_soft',p=FLEX),arm('ordinal_forest',kind='ordinal'),arm('ordinal_natural',kind='ordinal',w='natural')]
P2,S2bis=evaluate(M21,a,'02_calibration')
print('Baseline max absolute difference versus NB08:',np.max(abs(P2.baseline-OLD[BASE_NAME])))


## 03 — Spatial resolution 50/75/100/125 m
75/100 m targets from native NB08 reference; 125 m derived directly from NB07 1 m sampling, not averaging filtered cells. Reuse 10 m S2 and embeddings, 90% support per family. Different resolution cohorts; not a paired scale comparison. 250 m centre buffer for all scales in this stage.

In [ ]:
MOSAICS=defaultdict(list)
for item in M['mosaics']:
 p=NB07/'reconstructed_mosaics'/Path(item['output']).name
 with rasterio.open(p) as ds:MOSAICS[item['site']].append((item['name'],p,list(ds.bounds)))
for s in MOSAICS:MOSAICS[s].sort(key=lambda x:x[0])
TARGET125_DIR=RUN/'125m_tiles';TARGET125_DIR.mkdir(exist_ok=True)
# One bounded 1m lattice per tile and mosaic; all six scales derived from that lattice.
# A nearest-neighbour lattice is an area approximation, not exact native-pixel area.
def tile_bounds(tx,ty):return (tx*TILE_M,ty*TILE_M,(tx+1)*TILE_M,(ty+1)*TILE_M)
def tile_transform(tx,ty,spacing):
 x0,y0,x1,y1=tile_bounds(tx,ty);return from_origin(x0,y1,spacing,spacing)
def reduce_grid(a,scale,spacing):
 n=int(scale/spacing);h,w=a.shape;assert h%n==0 and w%n==0
 return a.reshape(h//n,n,w//n,n).sum(axis=(1,3))
TARGET_COLS=['site','mosaic','scale_m','row','col','x','y','target_fraction','mean_probability','uav_valid_fraction','tile_x','tile_y']
def target_tile(forest,tx,ty,opened):
 dest=TARGET125_DIR/f'{forest}_{tx}_{ty}.parquet'
 if dest.is_file():
  d=pd.read_parquet(dest);assert list(d.columns)==TARGET_COLS
  return d
 b=tile_bounds(tx,ty);tr=tile_transform(tx,ty,LATTICE_M);n=TILE_M//LATTICE_M
 best={}
 for scale in [125]:
  k=TILE_M//scale;best[scale]=(np.zeros((k,k),np.float32),np.full((k,k),np.nan,np.float32),np.full((k,k),np.nan,np.float32),np.full((k,k),-1,np.int16))
 for j,(name,p,bounds) in enumerate(MOSAICS[forest]):
  if not box(*b).intersects(box(*bounds)):continue
  ds=opened[name]
  with WarpedVRT(ds,crs=CRS,transform=tr,width=n,height=n,resampling=Resampling.nearest,nodata=-9999.) as vrt:
   z=vrt.read([1,2],masked=True)
  a=z.filled(-9999.).astype(np.float32,copy=False)
  valid=(~np.ma.getmaskarray(z).any(axis=0))&np.isfinite(a).all(axis=0)&(a[0]>=0)&(a[0]<=1)&((a[1]==0)|(a[1]==1))
  assert np.array_equal(a[1][valid],(a[0][valid]>=.5).astype(np.float32))
  if not valid.any():continue
  for scale in [125]:
   count=reduce_grid(valid.astype(np.int32),scale,LATTICE_M)
   support=count/float((scale/LATTICE_M)**2)
   total=reduce_grid(np.where(valid,a[1],0.),scale,LATTICE_M)
   prob=reduce_grid(np.where(valid,a[0],0.),scale,LATTICE_M)
   v,c,pb,idx=best[scale];take=(support>=MIN_SUPPORT)&(support>v)
   v[take]=support[take];c[take]=total[take]/count[take];pb[take]=prob[take]/count[take];idx[take]=j
 frames=[]
 for scale,(v,c,pb,idx) in best.items():
  rr,cc=np.where(idx>=0)
  if not len(rr):continue
  x=b[0]+(cc+.5)*scale;y=b[3]-(rr+.5)*scale
  frames.append(pd.DataFrame(dict(site=forest,mosaic=[MOSAICS[forest][j][0] for j in idx[rr,cc]],scale_m=scale,
   row=np.floor(-y/scale).astype('int64'),col=np.floor(x/scale).astype('int64'),x=x,y=y,target_fraction=c[rr,cc],mean_probability=pb[rr,cc],uav_valid_fraction=v[rr,cc],tile_x=tx,tile_y=ty)))
 d=pd.concat(frames,ignore_index=True)[TARGET_COLS] if frames else pd.DataFrame(columns=TARGET_COLS)
 assert not d.duplicated(['site','scale_m','row','col']).any()
 atomic_parquet(d,dest)
 return d

def aggregate_features(a,scale,count_indices=()):
 # Expand 10m pixels into equal 5m quadrants for exact aligned area weights
 # at 25/75m as well as integer multiples of 10m. No artificial new information.
 good=np.isfinite(a).all(axis=0)&(a!=-9999).all(axis=0)
 for i in count_indices:good &= a[i]>0
 base=np.repeat(np.repeat(good,2,axis=0),2,axis=1)
 n=scale//5;den=reduce_grid(base.astype(np.int32),scale,5)
 vals=[]
 for b in a:
  z=np.repeat(np.repeat(np.where(good,b,0.),2,axis=0),2,axis=1)
  sums=reduce_grid(z,scale,5)
  vals.append(np.divide(sums,den,out=np.full(sums.shape,np.nan,dtype=np.float32),where=den>0))
 return np.stack(vals),den/float(n*n)

def add_folds(d):
 d=d.copy();d['block21']=d.site+'_'+d.x.floordiv(500).astype(int).astype(str)+'_'+d.y.floordiv(500).astype(int).astype(str)
 d['fold21']=d.block21.map(lambda b:mapping.get(b,int(hashlib.sha256(b.encode()).hexdigest()[:8],16)%5)).astype(int)
 return d.sort_values(['site','row','col']).reset_index(drop=True)
def sat_array(f,tx,ty,fam):
 p=SAT_DIR/f'{f}_{tx}_{ty}_{fam}.tif'
 with rasterio.open(p) as ds:
  assert ds.tags().get('recipe')==SAT_ID and ds.transform.almost_equals(tile_transform(tx,ty,10)) and ds.shape==(150,150) and ds.crs==rasterio.crs.CRS.from_string(CRS)
  return ds.read()
def scale_table(k):
 if k==50:return M21.copy()
 dest=RUN/f'scale_{k}.parquet'
 if dest.is_file() and not FORCE_REBUILD:return pd.read_parquet(dest)
 t=pd.read_parquet(TARGET_DIR/'all_scales_uav_reference.parquet')
 if k==125:
  parts=[]
  for f,g in t.groupby('site'):
   with ExitStack() as stack:
    opened={name:stack.enter_context(rasterio.open(p)) for name,p,b in MOSAICS[f]}
    for tx,ty in g[['tile_x','tile_y']].drop_duplicates().itertuples(index=False,name=None):parts.append(target_tile(f,int(tx),int(ty),opened))
  t=pd.concat(parts,ignore_index=True)
 else:t=t[t.scale_m.eq(k)]
 parts=[]
 for (f,tx,ty),g in t.groupby(['site','tile_x','tile_y']):
  tx=int(tx);ty=int(ty);b=tile_bounds(tx,ty);rr=np.floor((b[3]-g.y.to_numpy())/k).astype(int);cc=np.floor((g.x.to_numpy()-b[0])/k).astype(int)
  vals=[];ok=np.ones(len(g),bool)
  for fam,names,counts in [('ANNUAL',ANNUAL,[14]),('EMB',EMB,[])]+[(f'Q{q}',QUARTER[(q-1)*11:q*11],[10]) for q in range(1,5)]:
   if not (SAT_DIR/f'{f}_{tx}_{ty}_{fam}.tif').is_file():ok[:]=False;break
   v,cov=aggregate_features(sat_array(f,tx,ty,fam),k,counts);vals.append(pd.DataFrame(v[:,rr,cc].T,columns=names,index=g.index));ok &= cov[rr,cc]>=MIN_SUPPORT-1e-6
  if ok.any():parts.append(pd.concat([g,*vals],axis=1).loc[ok])
 d=pd.concat(parts,ignore_index=True);d=add_folds(d[np.isfinite(d[BASE].to_numpy(float)).all(axis=1)]);atomic_parquet(d,dest);return d
def run_scale(k):
 z=scale_table(k);display(z.groupby('site').size())
 return evaluate(z,[arm('annual',ANNUAL[:14]),arm('combined'),arm('flex_natural',w='natural',p=FLEX),arm('flex_bins_soft',w='forest_bins_soft',p=FLEX)],f'03_scale_{k}m',250)
if RUN_SCALE:
 for k in [50,75,100,125]:optional(f'scale_{k}',lambda:run_scale(k))


## 04 — Spatial context at 50 m
Within-cell SD, 150/250 m neighbourhood means and SD from satellite rasters only. Halo from adjacent tiles; no reflected padding. 90% context support. Selected annual S2 features and first 16 embedding dimensions (fixed subset). Compare all arms at 360 m centre separation to prevent overlap of 250 m square footprints.

In [ ]:
CTX_NAMES=ANNUAL[:14]+EMB[:16]
def context_table():
 dest=RUN/'context_50m.parquet'
 if dest.is_file() and not FORCE_REBUILD:return pd.read_parquet(dest)
 parts=[];H=13;N=150+2*H
 for (f,tx,ty),g in M21.groupby(['site','tile_x','tile_y']):
  tx=int(tx);ty=int(ty);a=np.full((len(CTX_NAMES),N,N),np.nan,np.float32)
  for dx in [-1,0,1]:
   for dy in [-1,0,1]:
    xx=tx+dx;yy=ty+dy
    if not all((SAT_DIR/f'{f}_{xx}_{yy}_{fam}.tif').is_file() for fam in ['ANNUAL','EMB']):continue
    annual=sat_array(f,xx,yy,'ANNUAL');emb=sat_array(f,xx,yy,'EMB');z=np.concatenate([annual[:14],emb[:16]])
    good=np.isfinite(z).all(axis=0)&(z!=-9999).all(axis=0)&(annual[14]>0);z[:,~good]=np.nan
    r=H-dy*150;c=H+dx*150;r0=max(0,r);r1=min(N,r+150);c0=max(0,c);c1=min(N,c+150)
    if r1>r0 and c1>c0:a[:,r0:r1,c0:c1]=z[:,r0-r:r1-r,c0-c:c1-c]
  b=tile_bounds(tx,ty);rr=np.floor((b[3]-g.y.to_numpy())/10).astype(int)+H;cc=np.floor((g.x.to_numpy()-b[0])/10).astype(int)+H
  valid=np.isfinite(a).all(axis=0);a=np.nan_to_num(a,nan=0);values={}
  for size in [5,15,25]:
   cov=uniform_filter(valid.astype(np.float32),size=size,mode='constant');good=cov[rr,cc]>=.9-1e-6
   for j,name in enumerate(CTX_NAMES):
    m=uniform_filter(a[j],size=size,mode='constant')/np.maximum(cov,1e-8)
    v=uniform_filter(a[j]**2,size=size,mode='constant')/np.maximum(cov,1e-8)-m*m
    values[f'ctx{size*10}_sd_{name}']=np.where(good,np.sqrt(np.maximum(v[rr,cc],0)),np.nan)
    if size>5:values[f'ctx{size*10}_mean_{name}']=np.where(good,m[rr,cc],np.nan)
  parts.append(pd.concat([g,pd.DataFrame(values,index=g.index)],axis=1));print('CONTEXT',f,tx,ty,flush=True)
 d=pd.concat(parts,ignore_index=True);atomic_parquet(d,dest);return d
def run_context():
 global CT,CTX
 CT=context_table();CTX=[c for c in CT if c.startswith('ctx')];texture=[c for c in CTX if c.startswith('ctx50_')]
 z=CT.dropna(subset=CTX);display(z.groupby('site').size())
 return evaluate(z,[arm('baseline'),arm('texture',BASE+texture),arm('context',BASE+CTX),arm('context_flex',BASE+CTX,w='natural',p=FLEX)],'04_context',360)
if RUN_CONTEXT:optional('context',run_context)


## 05 — NICFI
Reuse the NB08 predictor table if valid; otherwise extract annual NICFI means and within-cell texture at 5 m. Historical 2021 Yoko vs 2020 UAV is a declared temporal mismatch. Compare on matched support.

In [ ]:
# Direct snapped 10 m Earth Engine downloads; no old predictor files.
import ee
EE_READY=False
def ensure_ee():
 global EE_READY
 if EE_READY:return
 try:ee.Initialize(project=EE_PROJECT)
 except Exception:
  ee.Authenticate()
  ee.Initialize(project=EE_PROJECT)
 EE_READY=True
def ee_region(tx,ty):
 b=transform_bounds(CRS,'EPSG:4326',*tile_bounds(tx,ty),densify_pts=21)
 return ee.Geometry.Rectangle(list(b),proj='EPSG:4326',geodesic=False)
NICFI_SOURCE='projects/planet-nicfi/assets/basemaps/africa'
NICFI_YEARS={**YEARS,'Yoko':2021}
NICFI_BANDS=['B','G','R','N']
NIC_CFG=dict(version='nicfi_v5_annual_5m_v1',source=NICFI_SOURCE,years=NICFI_YEARS,scale=5,crs=CRS,grid=[0,0],cadence='monthly',min_months=2,texture='within-cell SD and mean absolute neighboring NDVI differences; no cross-cell neighborhood',scale_factor=.0001)
NIC_ID=digest(NIC_CFG)[:20];NIC_DIR=RUN/'nicfi_5m'/NIC_ID;NIC_DIR.mkdir(parents=True,exist_ok=True);atomic_json(NIC_DIR/'config.json',NIC_CFG)
NIC_STATUS='incomplete';NIC_RUN=None
def nicfi_tile(forest,tx,ty):
 path=NIC_DIR/f'{forest}_{tx}_{ty}.tif';meta=path.with_suffix('.json')
 tf=tile_transform(tx,ty,5);n=TILE_M//5
 def verify():
  with rasterio.open(path) as ds:
   assert ds.shape==(n,n) and ds.count==4 and ds.crs==rasterio.crs.CRS.from_string(CRS) and ds.transform.almost_equals(tf) and ds.nodata==-9999
   assert list(ds.descriptions)==NICFI_BANDS and ds.tags().get('recipe')==NIC_ID
   return ds.read()
 if path.is_file() and meta.is_file():
  assert json.loads(meta.read_text())['recipe']==NIC_ID
  return verify()
 ensure_ee()
 year=NICFI_YEARS[forest]
 coll=ee.ImageCollection(NICFI_SOURCE).filterBounds(ee_region(tx,ty)).filterDate(f'{year}-01-01',f'{year+1}-01-01').filter(ee.Filter.eq('cadence','monthly'))
 ids=coll.aggregate_array('system:index').getInfo();assert len(ids)>=2,(forest,year,ids)
 im=coll.select(NICFI_BANDS).median().multiply(.0001).unmask(-9999,sameFootprint=False).toFloat()
 params=dict(crs=CRS,crs_transform=list(tf)[:6],dimensions=[n,n],format='GEO_TIFF',filePerBand=False)
 for attempt in range(4):
  try:
   resp=requests.get(im.getDownloadURL(params),timeout=(30,300));resp.raise_for_status()
   with MemoryFile(resp.content) as mem,mem.open() as ds:
    assert ds.shape==(n,n) and ds.count==4 and ds.crs==rasterio.crs.CRS.from_string(CRS) and ds.transform.almost_equals(tf)
    a=ds.read().astype(np.float32)
   assert np.isfinite(a).all()
   tmp=path.with_suffix('.partial.tif')
   with rasterio.open(tmp,'w',driver='GTiff',width=n,height=n,count=4,dtype='float32',crs=CRS,transform=tf,nodata=-9999,compress='deflate',tiled=True) as dst:
    dst.write(a);dst.descriptions=tuple(NICFI_BANDS);dst.update_tags(recipe=NIC_ID)
   os.replace(tmp,path);atomic_json(meta,dict(recipe=NIC_ID,source_image_ids=ids,year=year))
   return verify()
  except Exception as exc:
   if attempt==3:raise
   print('NICFI retry',forest,tx,ty,str(exc)[:160]);time.sleep(2**attempt)
NIC_MEAN=['NIC_'+b+'_mean' for b in NICFI_BANDS]+['NIC_NDVI_mean']
NIC_TEXTURE=['NIC_'+b+'_sd' for b in NICFI_BANDS]+['NIC_NDVI_sd','NIC_NDVI_neighbor_absdiff']

def run_nicfi():
 global DNIC,NIC_FEATURES
 NIC_FEATURES=NIC_MEAN+NIC_TEXTURE
 NIC_TABLE=RUN/f'nicfi_50m_{NIC_ID}.parquet'
 prior=ROOT/f'nicfi_50m_{SOURCE_SHA[:12]}_{NIC_ID}.parquet'
 if NIC_TABLE.is_file() and not FORCE_REBUILD:DNIC=pd.read_parquet(NIC_TABLE)
 elif prior.is_file() and not FORCE_REBUILD:
  DNIC=pd.read_parquet(prior)
  cols=KEYS+['x','y','target_fraction','fold21','block21']+BASE
  check=M21.merge(DNIC[KEYS],on=KEYS,validate='one_to_one')[cols].sort_values(KEYS).reset_index(drop=True)
  assert DNIC[cols].sort_values(KEYS).reset_index(drop=True).equals(check)
  assert np.isfinite(DNIC[NIC_FEATURES].to_numpy(float)).all()
  print('NB08 NICFI TABLE CACHE');atomic_parquet(DNIC,NIC_TABLE)
 else:
  parts=[]
  for (forest,tx,ty),g in M21.groupby(['site','tile_x','tile_y']):
   a=nicfi_tile(forest,int(tx),int(ty));b=tile_bounds(tx,ty)
   rr=np.floor((b[3]-g.y.to_numpy())/50).astype(int);cc=np.floor((g.x.to_numpy()-b[0])/50).astype(int)
   good=np.isfinite(a).all(axis=0)&(a!=-9999).all(axis=0)&(a>=0).all(axis=0)&((a[3]+a[2])>0)
   ndvi=np.divide(a[3]-a[2],a[3]+a[2],out=np.zeros_like(a[3]),where=good)
   stack=np.concatenate([a,ndvi[None]],axis=0)
   rows=[]
   for i,j in zip(rr,cc):
    z=stack[:,i*10:(i+1)*10,j*10:(j+1)*10];v=good[i*10:(i+1)*10,j*10:(j+1)*10]
    if v.mean()<.9:rows.append([np.nan]*11+[v.mean()]);continue
    q=z[:,v];nd=z[-1];dx=abs(np.diff(nd,axis=1))[v[:,1:]&v[:,:-1]];dy=abs(np.diff(nd,axis=0))[v[1:,:]&v[:-1,:]]
    rows.append([*q.mean(axis=1),*q.std(axis=1),np.concatenate([dx,dy]).mean(),v.mean()])
   f=pd.DataFrame(rows,columns=NIC_MEAN+NIC_TEXTURE+['nicfi_support'],index=g.index)
   parts.append(pd.concat([g,f],axis=1));print('NICFI tile',forest,tx,ty,flush=True)
  DNIC=pd.concat(parts).dropna(subset=NIC_MEAN+NIC_TEXTURE).reset_index(drop=True)
  atomic_parquet(DNIC,NIC_TABLE)

 display(DNIC.groupby('site').size())
 return evaluate(DNIC,[arm('baseline'),arm('NICFI_only',NIC_FEATURES),arm('NICFI_means',BASE+NIC_MEAN),arm('NICFI_texture',BASE+NIC_FEATURES),arm('NICFI_flex',BASE+NIC_FEATURES,w='natural',p=FLEX)],'05_NICFI')
if RUN_NICFI:optional('NICFI',run_nicfi)


## 06 — HLS temporal context
NASA HLS L30 + S30 v002, 30 m source, three retrospective calendar years ending in UAV acquisition year (same convention as NB08; not necessarily pre-acquisition). Harmonized narrow NIR, red, SWIR1/2; Fmask cloud, adjacency, shadow, snow, water and high aerosol excluded. Monthly pooled composites across three years, per-pixel temporal quantiles, amplitude and interannual differences. Native 30 m aligned download, exact 10 m quadrants for 50 m area aggregation; no new 10 m information. Matched-support baseline, temporal-only and combined. No HLS predictor claimed until extraction succeeds.

In [ ]:
HLS_IDS=['NASA/HLS/HLSL30/v002','NASA/HLS/HLSS30/v002']
HLS_IND=['NDVI','NDMI','NBR']
HLS_FEAT=[f'{b}_{q}' for b in HLS_IND for q in ['p10','p50','p90','amp','trend']]+[f'{b}_m{m:02d}' for b in HLS_IND for m in range(1,13)]
HLS_CFG=dict(version='hls_monthly_v1',source=HLS_IDS,years=YEARS,window='year-2 to year',mask='bits0-5 and high aerosol',grid=30,features=HLS_FEAT,min_months=8,aggregation='50m exact aligned 10m subcells')
HLS_ID=digest(HLS_CFG)[:20];HDIR=RUN/'hls'/HLS_ID;HDIR.mkdir(parents=True,exist_ok=True);atomic_json(HDIR/'config.json',HLS_CFG)
def hls_clean(im,s30):
 qa=im.select('Fmask');ok=qa.bitwiseAnd(63).eq(0).And(qa.rightShift(6).bitwiseAnd(3).neq(3))
 b=im.select(['B4','B8A','B11','B12'] if s30 else ['B4','B5','B6','B7'],['R','N','S1','S2']).toFloat()
 # Earth Engine catalog values may be stored as scaled integer or reflectance; ratios invariant.
 r,n,s1,s2=[b.select(c) for c in ['R','N','S1','S2']]
 def idx(x,y,name):return x.subtract(y).divide(x.add(y)).updateMask(x.add(y).gt(0)).rename(name)
 return ee.Image.cat([idx(n,r,'NDVI'),idx(n,s1,'NDMI'),idx(n,s2,'NBR')]).updateMask(ok).copyProperties(im,['system:time_start'])
def hls_tile(f,tx,ty):
 p=HDIR/f'{f}_{tx}_{ty}.tif';meta=p.with_suffix('.json');tf=tile_transform(tx,ty,30);names=HLS_FEAT+['valid_months']
 def read():
  with rasterio.open(p) as ds:
   assert ds.tags().get('recipe')==HLS_ID and ds.shape==(50,50) and list(ds.descriptions)==names and ds.transform.almost_equals(tf)
   return ds.read()
 if p.is_file() and meta.is_file():return read()
 ensure_ee();y=YEARS[f];area=ee_region(tx,ty)
 c1=ee.ImageCollection(HLS_IDS[0]).filterBounds(area).filterDate(f'{y-2}-01-01',f'{y+1}-01-01').map(lambda i:hls_clean(i,False))
 c2=ee.ImageCollection(HLS_IDS[1]).filterBounds(area).filterDate(f'{y-2}-01-01',f'{y+1}-01-01').map(lambda i:hls_clean(i,True))
 c=c1.merge(c2);ids=c.aggregate_array('system:index').getInfo();assert ids
 empty=ee.Image.constant([0,0,0]).rename(HLS_IND).updateMask(ee.Image.constant(0))
 def median(col):return col.merge(ee.ImageCollection.fromImages([empty])).median().select(HLS_IND)
 monthly=[median(c.filter(ee.Filter.calendarRange(m,m,'month'))) for m in range(1,13)]
 mc=ee.ImageCollection.fromImages(monthly);q=mc.reduce(ee.Reducer.percentile([10,50,90]));valid=mc.select('NDVI').count().unmask(0).rename('valid_months')
 trend=median(c.filterDate(f'{y}-01-01',f'{y+1}-01-01')).subtract(median(c.filterDate(f'{y-2}-01-01',f'{y-1}-01-01')))
 stack=[]
 for b in HLS_IND:
  stack += [q.select(b+'_p10'),q.select(b+'_p50'),q.select(b+'_p90'),q.select(b+'_p90').subtract(q.select(b+'_p10')).rename(b+'_amp'),trend.select(b).rename(b+'_trend')]
 for b in HLS_IND:
  # Missing individual months filled with pixel annual climatological median; support retained for QA.
  stack += [im.select(b).unmask(q.select(b+'_p50')).rename(f'{b}_m{m:02d}') for m,im in enumerate(monthly,1)]
 im=ee.Image.cat(stack+[valid]).select(names).unmask(-9999,sameFootprint=False).toFloat()
 params=dict(crs=CRS,crs_transform=list(tf)[:6],dimensions=[50,50],format='GEO_TIFF',filePerBand=False)
 for attempt in range(3):
  try:
   resp=requests.get(im.getDownloadURL(params),timeout=(30,300));resp.raise_for_status()
   with MemoryFile(resp.content) as mem,mem.open() as ds:
    assert ds.shape==(50,50) and ds.count==len(names) and ds.transform.almost_equals(tf);a=ds.read()
   tmp=p.with_suffix('.partial.tif')
   with rasterio.open(tmp,'w',driver='GTiff',width=50,height=50,count=len(names),dtype='float32',crs=CRS,transform=tf,nodata=-9999,compress='deflate') as ds:
    ds.write(a);ds.descriptions=tuple(names);ds.update_tags(recipe=HLS_ID)
   os.replace(tmp,p);atomic_json(meta,dict(recipe=HLS_ID,scene_ids=ids,year=y));return read()
  except Exception:
   if attempt==2:raise
   time.sleep(2**attempt)
def run_hls():
 global DHLS,HLS_FEATURES
 HLS_FEATURES=['HLS_'+c for c in HLS_FEAT]
 dest=RUN/f'hls_50m_{HLS_ID}.parquet'
 if dest.is_file() and not FORCE_REBUILD:DHLS=pd.read_parquet(dest)
 else:
  parts=[]
  for (f,tx,ty),g in M21.groupby(['site','tile_x','tile_y']):
   tx=int(tx);ty=int(ty);a=hls_tile(f,tx,ty);good=np.isfinite(a).all(axis=0)&(a!=-9999).all(axis=0)&(a[-1]>=8)
   v=np.repeat(np.repeat(good,3,axis=0),3,axis=1);den=reduce_grid(v.astype(int),50,10);b=tile_bounds(tx,ty)
   rr=np.floor((b[3]-g.y.to_numpy())/50).astype(int);cc=np.floor((g.x.to_numpy()-b[0])/50).astype(int);ok=den[rr,cc]>=23
   vals=[]
   for band in a[:-1]:
    z=np.repeat(np.repeat(np.where(good,band,0),3,axis=0),3,axis=1);s=reduce_grid(z,50,10)
    vals.append(s[rr,cc]/np.maximum(den[rr,cc],1))
   parts.append(pd.concat([g,pd.DataFrame(np.stack(vals,axis=1),columns=HLS_FEATURES,index=g.index)],axis=1).loc[ok]);print('HLS',f,tx,ty,flush=True)
  DHLS=pd.concat(parts,ignore_index=True);atomic_parquet(DHLS,dest)
 display(DHLS.groupby('site').size())
 return evaluate(DHLS,[arm('baseline'),arm('HLS_only',HLS_FEATURES),arm('HLS_summary',BASE+[c for c in HLS_FEATURES if '_m' not in c]),arm('HLS_full',BASE+HLS_FEATURES),arm('HLS_flex',BASE+HLS_FEATURES,w='natural',p=FLEX)],'06_HLS')
if RUN_HLS:optional('HLS',run_hls)


## 07 — Combined context ablation and synthesis
Only if all predictor tables are available; common complete-case support, 360 m buffer across all arms. Compare additions individually and together. No final model selected automatically.

In [ ]:
def combined():
 assert all(x in globals() for x in ['CT','CTX','DNIC','NIC_FEATURES','DHLS','HLS_FEATURES']),'Optional tables incomplete'
 keys=['site','row','col'];z=M21.merge(CT[keys+CTX],on=keys,validate='one_to_one').merge(DNIC[keys+NIC_FEATURES],on=keys,validate='one_to_one').merge(DHLS[keys+HLS_FEATURES],on=keys,validate='one_to_one').dropna(subset=CTX+NIC_FEATURES+HLS_FEATURES)
 display(z.groupby('site').size())
 return evaluate(z,[arm('baseline'),arm('context',BASE+CTX),arm('NICFI',BASE+NIC_FEATURES),arm('HLS',BASE+HLS_FEATURES),arm('all',BASE+CTX+NIC_FEATURES+HLS_FEATURES),arm('all_flex',BASE+CTX+NIC_FEATURES+HLS_FEATURES,w='natural',p=FLEX)],'07_combined',360)
if RUN_CONTEXT and RUN_NICFI and RUN_HLS:optional('combined',combined)
display(pd.DataFrame(REPORTS))
atomic_json(RUN/'exploration_status.json',dict(source=SOURCE_SHA,stages=REPORTS,exploratory=True,cleanup_authorized=False))
print('Use matched-support metrics, calibration plots and bootstrap intervals to decide final NB08 recipe.')


## 08 — Historical training recovery and target-definition sensitivity (NEW)
Run after 00–01 and the definition portion of 03 (needed for scale_table). No new satellite downloads. Existing blocks 02–07 and caches are unchanged. If restarting, set RUN_SCALE=False before executing 03 to restore its helpers without rerunning scale experiments. 50m and 100m use the same outer spatial block assignments as earlier stages; all arms within each resolution use a common 250m centre buffer for direct comparison with Block 03. A second 50m run uses the original 100m buffer to compare with Block 02. The historical HGB recipe is recovered from V3_1 NB04 Section 06: 500 cells/forest, four bins [0,10,20,40,100%], reserve 25% high cover then redistribute quotas, five independently resampled members, equal total forest weights, 200 iterations/20 leaves/40 min leaf/L2=2. Also test 1500 cap, 50% high quota, flexible settings and random ensembles. No synthetic duplication. Training uses only the outer training partition. Both binary fraction and mean UAV probability are predicted and scored against their own targets on identical cells. Mean probability is NOT claimed to be physical fractional cover. Compare target variance, paired metrics and calibration; do not choose a different biological target simply because its MAE is lower. Historical result: paired_random_vs_stratified_100m.csv, Drive file 1bTIyABCMhNSnNq0anQ75B3crvbpLMwc1. Historical mean-probability results are not direct V5 benchmarks.


In [ ]:
HIST=dict(learning_rate=.06,max_iter=200,max_leaf_nodes=20,min_samples_leaf=40,l2_regularization=2,early_stopping=False)
HIST_EDGES=np.array([0,.1,.2,.4,1.000001])
HIST_ARMS=[
 dict(name='all_baseline',cap=None,mode='all',quota=.25,members=1,params=MODEL),
 dict(name='all_flexible',cap=None,mode='all',quota=.25,members=1,params=FLEX),
 dict(name='random500',cap=500,mode='random',quota=.25,members=5,params=HIST),
 dict(name='strat500',cap=500,mode='strat',quota=.25,members=5,params=HIST),
 dict(name='random1500',cap=1500,mode='random',quota=.25,members=5,params=HIST),
 dict(name='strat1500',cap=1500,mode='strat',quota=.25,members=5,params=HIST),
 dict(name='strat500_high50',cap=500,mode='strat',quota=.5,members=5,params=HIST),
 dict(name='strat1500_high50',cap=1500,mode='strat',quota=.5,members=5,params=HIST),
 dict(name='strat1500_flex',cap=1500,mode='strat',quota=.25,members=5,params=FLEX)]
# Each five-member arm reports member 0 as well as ensemble mean, isolating ensemble effect.
# All-data controls use one member; paired random/strat arms share parameter settings.
def historical_draw(d,tr,target,cap,mode,quota,seed):
 rng=np.random.default_rng(seed);picked=[]
 for s in SITES:
  pool=tr[d.site.iloc[tr].eq(s).to_numpy()]
  assert len(pool)>0,('no training forest',s)
  if cap is None or len(pool)<=cap:chosen=pool.copy()
  elif mode=='random':chosen=np.sort(rng.choice(pool,cap,replace=False))
  else:
   y=d[target].to_numpy()[pool];b=np.clip(np.searchsorted(HIST_EDGES,y,side='right')-1,0,3)
   pools={k:pool[b==k] for k in range(4)};chosen=[]
   nh=min(len(pools[3]),int(np.ceil(cap*quota)))
   if nh:chosen.extend(rng.choice(pools[3],nh,replace=False).tolist())
   leftovers={k:np.setdiff1d(v,chosen) for k,v in pools.items()}
   while len(chosen)<cap:
    active=[k for k,v in leftovers.items() if len(v)]
    assert active
    per=max(1,(cap-len(chosen))//len(active))
    for k in active:
     m=min(per,len(leftovers[k]),cap-len(chosen))
     if m:
      p=rng.choice(leftovers[k],m,replace=False);chosen.extend(p.tolist());leftovers[k]=np.setdiff1d(leftovers[k],p)
   chosen=np.asarray(chosen,int)
  assert len(np.unique(chosen))==len(chosen) and np.isin(chosen,pool).all()
  picked.extend(chosen.tolist())
 return np.asarray(picked,int)
def historical_eval(d,scale,buffer,target):
 d=d.reset_index(drop=True).copy()
 assert d[target].between(0,1).all() and np.isfinite(d[BASE].to_numpy(float)).all()
 assert set(d.site)==set(SITES) and d.groupby('site').fold21.nunique().eq(5).all()
 cols=['site','row','col','x','y','block21','fold21','target_fraction','mean_probability']
 cfg=dict(version='historical_v1',source=dataframe_sha(d[cols+BASE]),scale=scale,buffer=buffer,target=target,features=BASE,arms=HIST_ARMS,seed=SEED21,sklearn=SK_VERSION)
 folder=RUN/f'08_historical_{scale}m_b{buffer}_{target}'/digest(cfg)[:20];folder.mkdir(parents=True,exist_ok=True);atomic_json(folder/'config.json',cfg)
 pred=d[cols].copy();names=[];draw_audit=[]
 for a in HIST_ARMS:
  names.append(a['name']+'_single')
  if a['members']>1:names.append(a['name']+'_ensemble')
 for k,tr,te in splits(d,buffer):
  for a in HIST_ARMS:
   name=a['name'];ps=[]
   for j in range(a['members']):
    seed=SEED21+10007*k+113*j
    ix=historical_draw(d,tr,target,a['cap'],a['mode'],a['quota'],seed)
    for s in SITES:
     ids=ix[d.site.iloc[ix].eq(s).to_numpy()];b=np.clip(np.searchsorted(HIST_EDGES,d[target].to_numpy()[ids],side='right')-1,0,3)
     draw_audit.append(dict(fold=k,model=name,member=j,site=s,n=len(ids),**{f'bin{h}':int((b==h).sum()) for h in range(4)}))
    p=folder/f'{name}_fold{k}_member{j}.npz'
    if p.is_file() and not FORCE_REBUILD:
     with np.load(p,allow_pickle=False) as z:
      assert np.array_equal(z['test'],te) and np.array_equal(z['train'],ix);v=z['pred']
    else:
     g=d.iloc[ix];w=weights(g,'forest')
     m=HistGradientBoostingRegressor(random_state=seed,**a['params'])
     m.fit(g[BASE].to_numpy(np.float32),g[target].to_numpy(np.float32),sample_weight=w)
     v=np.clip(m.predict(d.iloc[te][BASE].to_numpy(np.float32)),0,1)
     tmp=p.with_suffix('.partial')
     with open(tmp,'wb') as f:np.savez_compressed(f,test=te,train=ix,pred=v)
     os.replace(tmp,p)
    assert len(v)==len(te) and np.isfinite(v).all();ps.append(v)
   pred.loc[te,name+'_single']=ps[0]
   if len(ps)>1:pred.loc[te,name+'_ensemble']=np.mean(ps,axis=0)
   print('HIST',scale,buffer,target,k,name,flush=True)
 assert pred[names].notna().all().all();atomic_parquet(pred,folder/'predictions.parquet')
 pd.DataFrame(draw_audit).to_csv(folder/'training_draws.csv',index=False)
 rows=[];summ=[];cal=[];boot=[]
 for name in names:
  for s,g in pred.groupby('site'):
   y=g[target].to_numpy();p=g[name].to_numpy();e=p-y;h=y>=.4
   rows.append(dict(model=name,site=s,target_sd_pp=100*np.std(y),**stats(y,p),high40_n=int(h.sum()),high40_bias_pp=100*e[h].mean() if h.any() else np.nan,high40_MAE_pp=100*abs(e[h]).mean() if h.any() else np.nan,high40_recall=np.mean(p[h]>=.4) if h.any() else np.nan))
   for b,q in g.assign(bin=bins(y)).groupby('bin'):
    cal.append(dict(model=name,site=s,bin=b,n=len(q),observed_pp=100*q[target].mean(),predicted_pp=100*q[name].mean()))
  a=pd.DataFrame(rows);a=a[a.model.eq(name)];mos=a[a.site.eq('MOS')].iloc[0];u=stats(pred[target],pred[name]);den=np.mean([g[target].var(ddof=0) for _,g in pred.groupby('site')])
  summ.append(dict(model=name,n=len(pred),pooled_R2=u['R2'],pooled_MAE_pp=u['MAE_pp'],forest_MAE_pp=a.MAE_pp.mean(),within_forest_R2=1-np.mean((a.RMSE_pp/100)**2)/den,mean_site_R2=a.R2.mean(),mean_site_slope=a.slope.mean(),MOS_R2=mos.R2,MOS_MAE_pp=mos.MAE_pp,MOS_slope=mos.slope,MOS_high40_bias_pp=mos.high40_bias_pp,MOS_high60_bias_pp=mos.high_bias_pp,MOS_low10_bias_pp=mos.low_bias_pp))
  # Compare MAE and high/low bias to all-data baseline; resample within each forest.
  # Also compare stratified to random matched-cap ensemble/single when available.
  comparators=['all_baseline_single']
  for cap in [500,1500]:
   if name.startswith(f'strat{cap}_'):
    other=f'random{cap}_'+('ensemble' if name.endswith('_ensemble') else 'single')
    if other in names:comparators.append(other)
  for base in dict.fromkeys(comparators):
   rng=np.random.default_rng(SEED21);dist=[]
   for _,g in pred.groupby('site'):
    q=g.assign(delta=100*(abs(g[name]-g[target])-abs(g[base]-g[target]))).groupby('block21').delta.agg(['sum','count']).to_numpy()
    draws=q[rng.integers(0,len(q),(1000,len(q)))].sum(axis=1);dist.append(draws[:,0]/draws[:,1])
   v=np.mean(dist,axis=0)
   # MOS high/low signed bias differences; bootstrap may have no qualifying cells.
   z=pred[pred.site.eq('MOS')];entry=dict(model=name,baseline=base,delta_MAE_pp=np.mean([100*(abs(g[name]-g[target])-abs(g[base]-g[target])).mean() for _,g in pred.groupby('site')]),ci_low=np.quantile(v,.025),ci_high=np.quantile(v,.975))
   for label,mask in [('high40',z[target]>=.4),('high60',z[target]>=.6),('low10',z[target]<=.1)]:
    q=z[mask].assign(delta=100*(z.loc[mask,name]-z.loc[mask,base])).groupby('block21').delta.agg(['sum','count']).to_numpy()
    if len(q)>=5:
     dr=q[rng.integers(0,len(q),(1000,len(q)))].sum(axis=1);vv=dr[:,0]/dr[:,1]
     entry.update({label+'_delta_bias_pp':q[:,0].sum()/q[:,1].sum(),label+'_ci_low':np.quantile(vv,.025),label+'_ci_high':np.quantile(vv,.975),label+'_blocks':len(q)})
   boot.append(entry)
 sm=pd.DataFrame(summ).sort_values('forest_MAE_pp');pf=pd.DataFrame(rows);cb=pd.DataFrame(cal)
 sm.to_csv(folder/'summary.csv',index=False);pf.to_csv(folder/'by_forest.csv',index=False);cb.to_csv(folder/'calibration.csv',index=False);pd.DataFrame(boot).to_csv(folder/'paired_bootstrap.csv',index=False)
 display(sm.round(3));display(pf[pf.site.eq('MOS')].round(3))
 chosen=list(dict.fromkeys(['all_baseline_single','random500_ensemble','strat500_ensemble','strat1500_ensemble','strat1500_flex_ensemble',sm.iloc[0].model]))
 fig,axs=plt.subplots(2,len(chosen),figsize=(3.3*len(chosen),6.5),squeeze=False,layout='constrained')
 for j,name in enumerate(chosen):
  for i,g in enumerate([pred,pred[pred.site.eq('MOS')]]):
   ax=axs[i,j];ax.hexbin(100*g[target],100*g[name],gridsize=40,bins='log',mincnt=1,extent=(0,100,0,100));ax.plot([0,100],[0,100],'k--');ax.set(xlim=(0,100),ylim=(0,100),aspect='equal',xlabel='UAV reference (%)',ylabel='Held-out prediction (%)',title=('Pooled' if i==0 else 'MOS')+' | '+name)
 fig.savefig(folder/'pooled_MOS.png',dpi=150);plt.show();plt.close(fig)
 fig,axs=plt.subplots(2,2,figsize=(11,8),layout='constrained')
 for s,ax in zip(SITES,axs.ravel()):
  for name in chosen:
   q=cb[(cb.site==s)&(cb.model==name)];ax.plot(q.observed_pp,q.predicted_pp,'o-',label=name,ms=3)
  ax.plot([0,100],[0,100],'k--');ax.set(xlim=(0,100),ylim=(0,100),title=s,xlabel='Observed bin mean (%)',ylabel='Predicted bin mean (%)')
 axs[0,0].legend(fontsize=7);fig.savefig(folder/'calibration.png',dpi=150);plt.show();plt.close(fig)
 REPORTS.append(dict(stage=folder.parent.name,path=str(folder),n=len(pred),status='complete'))
 return pred,sm,pf
# 50m 100m buffer is the original operational comparison; 50/100m 250m buffer isolates scale under common exclusion.
HIST_RESULTS={}
for scale,buffer in [(50,100),(50,250),(100,250)]:
 d=scale_table(scale)
 assert 'mean_probability' in d.columns
 print('TARGET AUDIT',scale,buffer)
 display(d.groupby('site')[['target_fraction','mean_probability']].agg(['count','mean','std','min','max']).round(3))
 for target in ['target_fraction','mean_probability']:
  label=f'historical_{scale}_{buffer}_{target}'
  ans=optional(label,lambda:historical_eval(d,scale,buffer,target))
  if ans is not None:HIST_RESULTS[(scale,buffer,target)]=ans


## 09 — Matched target comparison and final decision evidence
Compare binary vs probability models on identical target locations. Report both each model's own-target accuracy and its agreement with binary cover; do not treat continuous probability predictions as binary cover estimates. Compare observed variance and per-forest correlation/offset. This is a diagnostic, not a conversion or calibrated correction.


In [ ]:
cross=[]
for scale,buffer in [(50,100),(50,250),(100,250)]:
 a=HIST_RESULTS.get((scale,buffer,'target_fraction'));b=HIST_RESULTS.get((scale,buffer,'mean_probability'))
 if a is None or b is None:continue
 pa,sa,_=a;pb,sb,_=b;cols=['site','row','col','x','y','fold21','target_fraction','mean_probability']
 assert pa[cols].equals(pb[cols])
 for s,g in pa.groupby('site'):
  print('TARGET',scale,buffer,s,'n',len(g),'binary SD pp',round(100*g.target_fraction.std(),2),'prob SD pp',round(100*g.mean_probability.std(),2),'correlation',round(g.target_fraction.corr(g.mean_probability),3),'prob minus binary pp',round(100*(g.mean_probability-g.target_fraction).mean(),2))
 for name in sa.model:
  for s,g in pa.groupby('site'):
   q=pb.loc[g.index]
   for training,p in [('binary',g[name]),('probability',q[name])]:
    for evaluation in ['target_fraction','mean_probability']:
     cross.append(dict(scale=scale,buffer=buffer,site=s,model=name,training_target=training,evaluation_target=evaluation,**stats(g[evaluation],p)))
  # same model, same data: compare own-target accuracy and binary accuracy of probability output
 if scale==50 and buffer==100:
  name='strat500_ensemble';fig,axs=plt.subplots(2,2,figsize=(10,9),layout='constrained')
  for i,(p,target) in enumerate([(pa,'target_fraction'),(pb,'mean_probability')]):
   for j,g in enumerate([p,p[p.site.eq('MOS')]]):
    ax=axs[i,j];ax.hexbin(100*g[target],100*g[name],gridsize=45,bins='log',mincnt=1,extent=(0,100,0,100));ax.plot([0,100],[0,100],'k--');ax.set(xlim=(0,100),ylim=(0,100),aspect='equal',xlabel=target+' (%)',ylabel='Prediction (%)',title=('Pooled' if j==0 else 'MOS')+' | '+target)
  fig.savefig(RUN/'09_matched_target_scatter.png',dpi=150);plt.show();plt.close(fig)
cross=pd.DataFrame(cross);cross.to_csv(RUN/'09_cross_target_metrics.csv',index=False)
if len(cross):display(cross[(cross.site=='MOS')&(cross.scale==50)&(cross.buffer==100)&(cross.model.isin(['all_baseline_single','strat500_ensemble','strat1500_ensemble','strat1500_flex_ensemble']))].round(3))
atomic_json(RUN/'historical_exploration_status.json',dict(source=SOURCE_SHA,stages=REPORTS,exploratory=True,cleanup_authorized=False))
print('Review paired bootstrap, low/high bias, calibration and both target definitions before changing NB08.')


## 10 — FINAL: nested spatial recalibration (50 m only)
Run 00 → 01 → 10; no need to run 02–09 again. Cache-only NICFI input; no Earth Engine requests. Four fixed models, no hyperparameter search. Reuse exact outer 100 m buffered held-out predictions from completed Blocks 02 and 05, verifying their config, data hash, fold and model specification. For each outer fold, split its eligible training data into two inner validation groups (pairs of the remaining four original spatial folds); exclude inner training centers within 100 m of inner validation centers by forest. Generate inner out-of-fold predictions exclusively from that outer training population. Calibrators fit on inner predictions only, using equal forest weights; optionally forest-specific calibration is deliberately NOT tested. Outer test data never enter calibration or inner fitting. Affine fits observed on prediction by weighted least squares; isotonic monotonic; quantile mapping matches weighted marginal distributions and is not a conditional-mean estimator. Quantile mapping may worsen point accuracy even if distribution improves. Report both objectives, including per-forest distribution Wasserstein distance, variance ratio, mean bias, high/low fractions, spatial block paired uncertainty. No automatic winner; repeated exploratory use of the same outer folds is not independent final confirmation. No change to NB08.


In [ ]:
from sklearn.isotonic import IsotonicRegression
from scipy.stats import wasserstein_distance
# Two inner groups keeps this final experiment focused; outer predictions are cached.
CAL_BUFFER=100
CAL_N_BOOT=1000
CAL_USE_ORDINAL=True  # set False only if runtime is prohibitive; reports reduced scope
CAL_FORCE_INNER=False
CAL_COLS=['site','row','col','x','y','target_fraction','block21','fold21']
NIC_COLS=['NIC_'+b+'_mean' for b in ['B','G','R','N']]+['NIC_NDVI_mean']+['NIC_'+b+'_sd' for b in ['B','G','R','N']]+['NIC_NDVI_sd','NIC_NDVI_neighbor_absdiff']
# Locate the completed V5 NICFI table; require exact reference match, no extraction fallback.
candidates10=sorted(RUN.glob('nicfi_50m_*.parquet'))
if not candidates10:
 candidates10=sorted(ROOT.glob(f'nicfi_50m_{SOURCE_SHA[:12]}_*.parquet'))
assert candidates10,'Missing cached NICFI table; finish Block 05 first'
base10=M21.reset_index(drop=True).copy()
valid10=[]
for path10 in candidates10:
 z=pd.read_parquet(path10)
 if not set(KEYS+NIC_COLS+CAL_COLS+BASE).issubset(z.columns):continue
 if z.duplicated(KEYS).any():continue
 z=base10[KEYS].merge(z,on=KEYS,how='left',validate='one_to_one')
 if z[CAL_COLS+BASE].equals(base10[CAL_COLS+BASE]) and np.isfinite(z[NIC_COLS].to_numpy(float)).all():valid10.append((path10,z))
assert valid10,'No complete exact-match NICFI cache'
assert len({dataframe_sha(z[KEYS+NIC_COLS]) for _,z in valid10})==1,'Conflicting NICFI caches'
nic_path10,nic10=valid10[0]
d10=base10.copy()
for c in NIC_COLS:d10[c]=nic10[c].to_numpy()
models10=[('baseline','02_calibration','baseline',BASE,'reg','forest',MODEL),
 ('flex_natural','02_calibration','flex_natural',BASE,'reg','natural',FLEX),
 ('NICFI_flex','05_NICFI','NICFI_flex',BASE+NIC_COLS,'reg','natural',FLEX)]
if CAL_USE_ORDINAL:models10.append(('ordinal_forest','02_calibration','ordinal_forest',BASE,'ordinal','forest',MODEL))
# Cache verification: recreate evaluator identity from its full configuration and actual data.
def outer_cache10(stage,oldname,features,kind,w,params):
 matches=[]
 for p in sorted((RUN/stage).glob('*/predictions.parquet')):
  cfg=json.loads((p.parent/'config.json').read_text())
  if cfg.get('version')!='08bis_cv_v1' or cfg.get('buffer')!=CAL_BUFFER or cfg.get('seed')!=SEED21 or cfg.get('sklearn')!=SK_VERSION:continue
  if digest(cfg)[:20]!=p.parent.name:continue
  if [oldname,list(features),kind,w,dict(params)] not in cfg['arms']:continue
  fields=sorted({c for _,f,_,_,_ in cfg['arms'] for c in f})
  if not set(fields).issubset(d10.columns):continue
  if dataframe_sha(d10[CAL_COLS+fields])!=cfg['source']:continue
  z=pd.read_parquet(p)
  if z[CAL_COLS].equals(d10[CAL_COLS]) and oldname in z and np.isfinite(z[oldname]).all():matches.append((p,z[oldname].to_numpy()))
 assert matches,('No verified outer prediction cache',stage,oldname)
 assert all(np.array_equal(v,matches[0][1]) for _,v in matches),'Conflicting predictions'
 return matches[0]
outer10={name:outer_cache10(stage,oldname,f,kind,w,p) for name,stage,oldname,f,kind,w,p in models10}
assert np.allclose(outer10['baseline'][1],OLD[BASE_NAME].to_numpy(),atol=1e-8,rtol=0),'Baseline mismatch'
cfg10=dict(version='nested_calibration_v1',source=dataframe_sha(d10[CAL_COLS+BASE+NIC_COLS]),outer={name:dict(path=str(path),hash=dataframe_sha(pd.DataFrame({'p':v}))) for name,(path,v) in outer10.items()},models=models10,buffer=CAL_BUFFER,inner_groups=2,seed=SEED21,sklearn=SK_VERSION,calibration_weight='equal_forest',bootstrap=CAL_N_BOOT)
dir10=RUN/'10_final_nested_calibration'/digest(cfg10)[:20];dir10.mkdir(parents=True,exist_ok=True);atomic_json(dir10/'config.json',cfg10)
pred10=d10[CAL_COLS].copy();aud10=[];param10=[]
def inner_split10(tr,k):
 remaining=[j for j in range(5) if j!=k]
 # Existing spatial blocks, no random pixel split; complete inner OOF coverage.
 for j,folds in enumerate([remaining[::2],remaining[1::2]]):
  va=tr[np.isin(d10.fold21.to_numpy()[tr],folds)];pool=tr[~np.isin(d10.fold21.to_numpy()[tr],folds)]
  xy=d10[['x','y']].to_numpy();site=d10.site.to_numpy();keep=np.ones(len(pool),bool)
  for s in SITES:
   a=np.flatnonzero(site[pool]==s);b=va[site[va]==s]
   if len(a) and len(b):keep[a]=cKDTree(xy[b]).query(xy[pool[a]])[0]>=CAL_BUFFER
  it=pool[keep]
  assert len(it)>0 and len(va)>0 and not np.intersect1d(it,va).size
  assert set(site[it])==set(SITES),('Missing forest in inner training',k,j)
  yield j,it,va
def weighted_cdf10(x,w):
 order=np.argsort(x,kind='stable');x=np.asarray(x)[order];w=np.asarray(w)[order]
 u,idx=np.unique(x,return_index=True);mass=np.add.reduceat(w,idx);cdf=(np.cumsum(mass)-.5*mass)/mass.sum()
 return u,cdf
def calibrate10(x,y,w,test):
 # Direction is observed ~ predicted, NOT inverse of compression slope.
 mu=np.average(x,weights=w);nu=np.average(y,weights=w);var=np.average((x-mu)**2,weights=w)
 slope=max(0.,np.average((x-mu)*(y-nu),weights=w)/var) if var>1e-12 else 0.
 intercept=nu-slope*mu
 affine=np.clip(intercept+slope*test,0,1)
 iso=IsotonicRegression(y_min=0,y_max=1,out_of_bounds='clip').fit(x,y,sample_weight=w)
 px,pc=weighted_cdf10(x,w);yx,yc=weighted_cdf10(y,w)
 q=np.interp(test,px,pc,left=0,right=1);qm=np.interp(q,yc,yx,left=yx[0],right=yx[-1])
 return {'raw':test,'affine':affine,'isotonic':iso.predict(test),'quantile':np.clip(qm,0,1)},dict(affine_intercept=intercept,affine_slope=slope,isotonic_x=iso.X_thresholds_.tolist(),isotonic_y=iso.y_thresholds_.tolist(),quantile_pred_x=px.tolist(),quantile_pred_cdf=pc.tolist(),quantile_target_x=yx.tolist(),quantile_target_cdf=yc.tolist())
for k,tr,te in splits(d10,CAL_BUFFER):
 for name,stage,oldname,f,kind,w,p in models10:
  inner=np.full(len(d10),np.nan)
  for j,it,va in inner_split10(tr,k):
   assert not np.intersect1d(te,it).size and not np.intersect1d(te,va).size
   cache=dir10/f'{name}_outer{k}_inner{j}.npz'
   if cache.is_file() and not CAL_FORCE_INNER:
    with np.load(cache,allow_pickle=False) as z:
     assert np.array_equal(z['train'],it) and np.array_equal(z['val'],va);v=z['pred']
   else:
    v=fit(d10,it,va,f,kind,w,p)
    tmp=cache.with_suffix('.partial')
    with open(tmp,'wb') as out:np.savez_compressed(out,train=it,val=va,pred=v)
    os.replace(tmp,cache)
   assert np.isfinite(v).all();inner[va]=v
   for s in SITES:aud10.append(dict(outer=k,inner=j,model=name,site=s,n_train=int(d10.site.iloc[it].eq(s).sum()),n_cal=int(d10.site.iloc[va].eq(s).sum())))
   print('CAL',name,'outer',k,'inner',j,flush=True)
  assert np.isfinite(inner[tr]).all() and np.isnan(inner[te]).all()
  g=d10.iloc[tr];cw=weights(g,'forest');yp=g.target_fraction.to_numpy()
  result,par=calibrate10(inner[tr],yp,cw,outer10[name][1][te])
  for method,v in result.items():pred10.loc[te,name+'__'+method]=v
  param10.append(dict(model=name,outer=k,n_cal=len(tr),**par))
  atomic_json(dir10/f'{name}_outer{k}_calibrator.json',param10[-1])
names10=[name+'__'+method for name,*_ in models10 for method in ['raw','affine','isotonic','quantile']]
assert pred10[names10].notna().all().all()
atomic_parquet(pred10,dir10/'heldout_predictions.parquet');pd.DataFrame(aud10).to_csv(dir10/'inner_support.csv',index=False)
rows10=[];sum10=[];boot10=[];bins10=[]
for name in names10:
 for s,g in pred10.groupby('site'):
  y=g.target_fraction.to_numpy();p=g[name].to_numpy()
  rows10.append(dict(model=name,site=s,**stats(y,p),sd_ratio=np.std(p)/np.std(y) if np.std(y)>0 else np.nan,wasserstein_pp=100*wasserstein_distance(y,p),obs_high40_share=np.mean(y>=.4),pred_high40_share=np.mean(p>=.4),obs_high60_share=np.mean(y>=.6),pred_high60_share=np.mean(p>=.6),obs_low10_share=np.mean(y<=.1),pred_low10_share=np.mean(p<=.1)))
  for b,q in g.assign(bin=bins(y)).groupby('bin'):bins10.append(dict(model=name,site=s,bin=b,n=len(q),observed_pp=100*q.target_fraction.mean(),predicted_pp=100*q[name].mean()))
 a=pd.DataFrame(rows10);a=a[a.model.eq(name)];mos=a[a.site.eq('MOS')].iloc[0];u=stats(pred10.target_fraction,pred10[name]);den=np.mean([g.target_fraction.var(ddof=0) for _,g in pred10.groupby('site')])
 sum10.append(dict(model=name,pooled_R2=u['R2'],pooled_MAE_pp=u['MAE_pp'],forest_MAE_pp=a.MAE_pp.mean(),within_forest_R2=1-np.mean((a.RMSE_pp/100)**2)/den,forest_Wasserstein_pp=a.wasserstein_pp.mean(),MOS_R2=mos.R2,MOS_MAE_pp=mos.MAE_pp,MOS_slope=mos.slope,MOS_high_bias_pp=mos.high_bias_pp,MOS_low_bias_pp=mos.low_bias_pp,MOS_sd_ratio=mos.sd_ratio,MOS_Wasserstein_pp=mos.wasserstein_pp,MOS_high60_share_error_pp=100*(mos.pred_high60_share-mos.obs_high60_share),MOS_low10_share_error_pp=100*(mos.pred_low10_share-mos.obs_low10_share)))
 for base in dict.fromkeys([name.split('__')[0]+'__raw','baseline__raw']):
  rng=np.random.default_rng(SEED21);dist=[]
  for s,g in pred10.groupby('site'):
   q=g.assign(delta=100*(abs(g[name]-g.target_fraction)-abs(g[base]-g.target_fraction))).groupby('block21').delta.agg(['sum','count']).to_numpy()
   dr=q[rng.integers(0,len(q),(CAL_N_BOOT,len(q)))].sum(axis=1);dist.append(dr[:,0]/dr[:,1])
  v=np.mean(dist,axis=0);entry=dict(model=name,baseline=base,delta_forest_MAE_pp=np.mean([100*(abs(g[name]-g.target_fraction)-abs(g[base]-g.target_fraction)).mean() for _,g in pred10.groupby('site')]),ci_low=np.quantile(v,.025),ci_high=np.quantile(v,.975))
  z=pred10[pred10.site.eq('MOS')]
  for label,mask in [('high60',z.target_fraction>=.6),('low10',z.target_fraction<=.1)]:
   q=z[mask].assign(delta=100*(z.loc[mask,name]-z.loc[mask,base])).groupby('block21').delta.agg(['sum','count']).to_numpy()
   if len(q)>=5:
    dr=q[rng.integers(0,len(q),(CAL_N_BOOT,len(q)))].sum(axis=1);v=dr[:,0]/dr[:,1]
    entry.update({label+'_delta_bias_pp':q[:,0].sum()/q[:,1].sum(),label+'_ci_low':np.quantile(v,.025),label+'_ci_high':np.quantile(v,.975)})
  boot10.append(entry)
summary10=pd.DataFrame(sum10).sort_values('forest_MAE_pp');per10=pd.DataFrame(rows10);cb10=pd.DataFrame(bins10);ci10=pd.DataFrame(boot10)
for df,f in [(summary10,'summary.csv'),(per10,'by_forest.csv'),(cb10,'calibration_bins.csv'),(ci10,'paired_bootstrap.csv')]:df.to_csv(dir10/f,index=False)
display(summary10.round(3));display(ci10.round(3))
mos=pred10[pred10.site.eq('MOS')]
fig,axs=plt.subplots(len(models10),4,figsize=(14,3.4*len(models10)),squeeze=False,layout='constrained')
for i,(name,*_) in enumerate(models10):
 for j,method in enumerate(['raw','affine','isotonic','quantile']):
  ax=axs[i,j];col=name+'__'+method
  ax.hexbin(100*mos.target_fraction,100*mos[col],gridsize=40,bins='log',mincnt=1,extent=(0,100,0,100));ax.plot([0,100],[0,100],'k--');ax.set(xlim=(0,100),ylim=(0,100),aspect='equal',title=name+' | '+method,xlabel='UAV cover (%)',ylabel='Prediction (%)')
fig.savefig(dir10/'MOS_calibration_scatter.png',dpi=150);plt.show();plt.close(fig)
fig,axs=plt.subplots(len(models10),2,figsize=(11,4*len(models10)),squeeze=False,layout='constrained')
for i,(name,*_) in enumerate(models10):
 ax=axs[i,0];ax.plot([0,100],[0,100],'k--')
 ax2=axs[i,1];grid=np.linspace(0,100,101);ax2.plot(grid,np.searchsorted(np.sort(100*mos.target_fraction),grid,side='right')/len(mos),'k--',label='UAV')
 for method in ['raw','affine','isotonic','quantile']:
  col=name+'__'+method;q=cb10[(cb10.site=='MOS')&(cb10.model==col)]
  ax.plot(q.observed_pp,q.predicted_pp,'o-',label=method,ms=3);ax2.plot(grid,np.searchsorted(np.sort(100*mos[col]),grid,side='right')/len(mos),label=method)
 ax.set(xlim=(0,100),ylim=(0,100),title=name+' | conditional bias',xlabel='Observed bin mean (%)',ylabel='Predicted bin mean (%)');ax.legend(fontsize=8)
 ax2.set(title=name+' | marginal CDF',xlabel='Cover (%)',ylabel='Cumulative fraction');ax2.legend(fontsize=8)
fig.savefig(dir10/'MOS_conditional_and_marginal.png',dpi=150);plt.show();plt.close(fig)
atomic_json(dir10/'status.json',dict(complete=True,n=len(pred10),models=[x[0] for x in models10],scope='exploratory',outer_cached=True,NB08_modified=False))
print('FINAL NB08bis outputs:',dir10)
print('Choose point prediction and distribution objectives separately. Review paired intervals before finalizing NB08; no further model grid planned.')
